# PERFORMANCE

In [0]:
# ============================================================
# 10_PERFORMANCE_OPTIMIZATION
#
# Purpose:
# Learn Spark and Delta Lake performance tuning
#
# Topics:
# - Partitioning
# - OPTIMIZE
# - ZORDER
# - Broadcast joins
# - Cache
# - Explain plans
# - AQE
# - Repartition
# - Coalesce
#
# ============================================================


from pyspark.sql.functions import *
from pyspark.sql.functions import broadcast
from delta.tables import DeltaTable



# ============================================================
# 1. Configuration
# ============================================================


orders_table = "ecommerce.silver.orders"

customers_table = "ecommerce.silver.customers"

gold_table = "ecommerce.gold.performance_sales"





# ============================================================
# 2. Check Current Spark Configuration
#
# Spark has many optimization settings
#
# ============================================================


print("Spark AQE Status")


spark.conf.get(
    "spark.sql.adaptive.enabled"
)





# ============================================================
# 3. Enable Adaptive Query Execution
#
# AQE automatically:
#
# - changes join strategies
# - reduces shuffle partitions
# - handles skew
#
# ============================================================


spark.conf.set(
    "spark.sql.adaptive.enabled",
    "true"
)


print(
    "AQE Enabled"
)





# ============================================================
# 4. Read Tables
# ============================================================


orders = spark.table(
    orders_table
)


customers = spark.table(
    customers_table
)





# ============================================================
# 5. Explain Query Plan
#
# Shows:
#
# - Scan
# - Join
# - Shuffle
# - Exchange
#
# ============================================================



join_df = (

    orders

    .join(
        customers,
        "customer_id"
    )

)



print(
    "Execution Plan"
)


join_df.explain(
    True
)





# ============================================================
# 6. Broadcast Join
#
# Use when one table is small
#
# Instead of shuffling customers
# to every executor,
# send customers table directly.
#
# ============================================================



broadcast_join = (

    orders

    .join(

        broadcast(customers),

        "customer_id"

    )

)



broadcast_join.explain(
    True
)



print(
    "Broadcast Join Applied"
)





# ============================================================
# 7. Cache
#
# Useful when same dataframe
# is used multiple times
#
# ============================================================



customers.cache()



print(
    "Customers cached"
)



customers.count()



print(
    "Cache materialized"
)





# Remove cache later

customers.unpersist()


print(
    "Cache removed"
)





# ============================================================
# 8. Repartition
#
# Increases partitions
#
# Useful before:
#
# - large writes
# - heavy transformations
#
# ============================================================



orders_repartitioned = (

    orders

    .repartition(
        8,
        "customer_id"
    )

)



print(
    "Partitions after repartition:"
)


print(
    orders_repartitioned.rdd.getNumPartitions()
)





# ============================================================
# 9. Coalesce
#
# Decreases partitions
#
# Useful before writing small datasets
#
# ============================================================



orders_small = (

    orders

    .coalesce(
        2
    )

)



print(
    "Partitions after coalesce:"
)


print(
    orders_small.rdd.getNumPartitions()
)





# ============================================================
# 10. Create Partitioned Delta Table
#
# Partitioning helps when filtering
#
# Example:
#
# WHERE order_date='2026-01-01'
#
# Spark reads only required partition
#
# ============================================================



partitioned_orders = (

    orders

    .withColumn(

        "year",

        year(
            col("order_date")
        )

    )


)



partitioned_orders.write \
.format("delta") \
.mode("overwrite") \
.partitionBy(
    "year"
) \
.saveAsTable(
    gold_table
)



print(
    "Partitioned table created"
)





# ============================================================
# 11. Check Table Files
#
# Delta stores data files
#
# ============================================================



spark.sql(
f"""
DESCRIBE DETAIL {gold_table}
"""
).show(
    truncate=False
)





# ============================================================
# 12. OPTIMIZE
#
# Combines small files
#
# ============================================================



spark.sql(
f"""
OPTIMIZE {gold_table}
"""
)



print(
    "OPTIMIZE completed"
)





# ============================================================
# 13. ZORDER
#
# Improves data skipping
#
# Best for columns frequently filtered
#
# ============================================================



spark.sql(
f"""
OPTIMIZE {gold_table}
ZORDER BY(customer_id)
"""
)



print(
    "ZORDER completed"
)





# ============================================================
# 14. Data Skipping Example
#
# Query filtered data
#
# ============================================================



result = spark.sql(
f"""
SELECT *
FROM {gold_table}
WHERE customer_id = 100
"""
)



display(
    result
)





# ============================================================
# 15. File Size Analysis
#
# Helps detect:
#
# Small file problem
#
# ============================================================



spark.sql(
f"""
DESCRIBE DETAIL {gold_table}
"""
).select(
    "numFiles",
    "sizeInBytes"
).show()





# ============================================================
# 16. Cleanup
# ============================================================



print(
    "Performance notebook completed successfully"
)

Spark AQE Status


---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8830453143573451>, line 53
     42 # ============================================================
     43 # 2. Check Current Spark Configuration
     44 #
     45 # Spark has many optimization settings
     46 #
     47 # ============================================================
     50 print("Spark AQE Status")
---> 53 spark.conf.get(
     54     "spark.sql.adaptive.enabled"
     55 )
     61 # ============================================================
     62 # 3. Enable Adaptive Query Execution
     63 #
   (...)
     69 #
     70 # ============================================================
     73 spark.conf.set(
     74     "spark.sql.adaptive.enabled",
     75     "true"
     76 )

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/conf.py:90, in RuntimeConf.get(self, key, default)
     8